In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/superstore.csv', encoding='latin1')

Initial EDA

In [3]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [5]:
df.nunique()

Row ID           9994
Order ID         5009
Order Date       1237
Ship Date        1334
Ship Mode           4
Customer ID       793
Customer Name     793
Segment             3
Country             1
City              531
State              49
Postal Code       631
Region              4
Product ID       1862
Category            3
Sub-Category       17
Product Name     1850
Sales            5825
Quantity           14
Discount           12
Profit           7287
dtype: int64

In [6]:
df["Segment"].unique() 

array(['Consumer', 'Corporate', 'Home Office'], dtype=object)

In [7]:
df["Category"].unique() 

array(['Furniture', 'Office Supplies', 'Technology'], dtype=object)

In [8]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [9]:
df.duplicated().sum()

np.int64(0)

Feature Engineering & Mapping

In [10]:
df_cleaned = df.copy()

In [11]:
# Chuẩn hóa dữ liệu thời gian (Datetime Parsing)
df_cleaned['Order Date'] = pd.to_datetime(df_cleaned['Order Date'], 
                                          format='%m/%d/%Y', errors='coerce')

df_cleaned['Ship Date'] = pd.to_datetime(df_cleaned['Ship Date'], 
                                         format='%m/%d/%Y', errors='coerce')

In [12]:
# Mô phỏng kênh phân phối (Channel Mapping)
# DTC : Direct-to-Customer (Bán hàng trực tiếp cho khách hàng)
# B2B : Business-to-Business (Bán hàng cho doanh nghiệp)
channel_mapping = {
    'Consumer': 'DTC',
    'Corporate': 'B2B',
    'Home Office': 'B2B'
}
df_cleaned['Sales_Channel'] = df_cleaned['Segment'].map(channel_mapping)

In [13]:
# Mô phỏng Ngành hàng FMCG (Product Line Mapping)
# Ánh xạ dựa trên giả định về vòng đời sản phẩm và tần suất mua lặp lại
# Liquid Milk: Sữa nước (Mua hàng thường xuyên, vòng đời ngắn)
# Yogurt & Desserts: Sữa chua & Tráng miệng (Mua hàng định kỳ, vòng đời trung bình)
# Powdered & Nutrition: Sữa bột & Dinh dưỡng (Mua hàng ít thường xuyên, vòng đời dài)
category_mapping = {
    'Furniture': 'Liquid Milk',
    'Office Supplies': 'Yogurt & Desserts',
    'Technology': 'Powdered & Nutrition'
}
df_cleaned['FMCG_Category'] = df_cleaned['Category'].map(category_mapping)

In [14]:
# Trích xuất Đặc trưng Thời gian (Temporal Features)
df_cleaned['Order_Month'] = df_cleaned['Order Date'].dt.month
df_cleaned['Order_Year'] = df_cleaned['Order Date'].dt.year
df_cleaned['Order_Quarter'] = df_cleaned['Order Date'].dt.quarter

In [15]:
# Đặc trưng Hành vi Tài chính & Rủi ro (Financial & Risk Behaviors)

# 1. Đo lường tốc độ giao hàng (Lead Time)
df_cleaned['Delivery_Time_Days'] = (df_cleaned['Ship Date'] - df_cleaned['Order Date']).dt.days

# 2. Tính Lợi nhuận biên (Profit Margin)
df_cleaned['Profit_Margin'] = np.where(df_cleaned['Sales'] > 0, 
                                         (df_cleaned['Profit'] / df_cleaned['Sales']), 
                                         0)
df_cleaned['Profit_Margin'] = df_cleaned['Profit_Margin'].round(2)

# 3. Cắm cờ "Nghiện chiết khấu" (High Discount Flag)
# Hỗ trợ phân tích tập khách hàng: Nếu discount > 15%, đánh cờ rủi ro (1)
df_cleaned['High_Discount_Flag'] = df_cleaned['Discount'].apply(lambda x: 1 if x > 0.15 else 0)

In [16]:
cols_to_drop = ['Row ID', 'Segment', 'Category']
df_cleaned = df_cleaned.drop(columns=cols_to_drop)

In [17]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order ID            9994 non-null   object        
 1   Order Date          9994 non-null   datetime64[ns]
 2   Ship Date           9994 non-null   datetime64[ns]
 3   Ship Mode           9994 non-null   object        
 4   Customer ID         9994 non-null   object        
 5   Customer Name       9994 non-null   object        
 6   Country             9994 non-null   object        
 7   City                9994 non-null   object        
 8   State               9994 non-null   object        
 9   Postal Code         9994 non-null   int64         
 10  Region              9994 non-null   object        
 11  Product ID          9994 non-null   object        
 12  Sub-Category        9994 non-null   object        
 13  Product Name        9994 non-null   object      

In [18]:
df_cleaned.to_csv('../data/sales_cleaned.csv', index=False)